# 04 — Semantic Search & Recommendation
### DiscoverAI · Deloitte x LUISS

Full search pipeline on top of the FAISS index built in notebook 03.

**Three search functions — run sections in order:**
- `search()` — base retrieval with quality re-ranking (section 3)
- `search_v2()` — adds negation filter: 'without X', 'X-free' (section 8)
- `search_v3()` — adds min_rating, dosage filter, synonym expansion (section 10)

**Score formula:**
```
score = similarity + beta_quality * quality_score + beta_pop * popularity_score
```
beta_quality is adaptive: 0.20 when similarity < 0.65, 0.12 otherwise.


## 0 . Setup

Loads FAISS index, product profiles, and MPNet model.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import faiss
import re
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

DATA_DIR = "/Users/danielegiovanardi/MSTERRORS/Mean-Squared-Terrors/data"

# Carica tutti i file prodotti dall'embedding
index_df  = pd.read_csv(os.path.join(DATA_DIR, "embedding_index.csv"))
comb_emb  = np.load(os.path.join(DATA_DIR, "combined_embeddings.npy"))
index     = faiss.read_index(os.path.join(DATA_DIR, "faiss_index.bin"))

# Carica review per il sentiment aggregato
reviews   = pd.read_csv(os.path.join(DATA_DIR, "reviews_cleaned.csv"))
products  = pd.read_csv(os.path.join(DATA_DIR, "products_cleaned.csv"))

# Carica modello di embedding
import torch
if torch.backends.mps.is_available():   DEVICE = "mps"
elif torch.cuda.is_available():          DEVICE = "cuda"
else:                                    DEVICE = "cpu"

model = SentenceTransformer("all-mpnet-base-v2", device=DEVICE)
print(f"Device: {DEVICE}")
print(f"Prodotti indicizzati: {index.ntotal:,}")
print(f"Colonne index: {list(index_df.columns)}")


## 1 . Pre-compute quality scores

Computed once per session. Never recomputed at query time.

- `quality_score` = 0.40 * rating_norm + 0.35 * sentiment + 0.25 * helpful_credibility
- `popularity_score` = log-normalised total review count


In [ ]:
# Aggregate review signals per product
rev_agg = reviews.groupby("parent_asin").agg(
    n_reviews      = ("rating", "count"),
    avg_rating_rev = ("rating", "mean"),
    pct_positive   = ("rating", lambda x: (x >= 4).mean()),
    pct_negative   = ("rating", lambda x: (x <= 2).mean()),
    total_helpful  = ("helpful_vote", "sum"),
    max_helpful    = ("helpful_vote", "max"),
).reset_index()

index_df = index_df.merge(rev_agg, on="parent_asin", how="left")

# Quality score ─────────────────────────────────────────────────────────────
# 1. Rating normalised to [0,1]
rating_norm = (index_df["product_avg_rating"].fillna(3.0) - 1) / 4

# 2. Sentiment
sentiment = index_df["pct_positive"].fillna(0.5) - 0.5 * index_df["pct_negative"].fillna(0.5)

# 3. Helpful credibility
hv_log = np.log1p(index_df["total_helpful"].fillna(0))
hv_p95 = hv_log.quantile(0.95)
helpful_cred = (hv_log / hv_p95).clip(upper=1.0)

# Weighted combination
index_df["quality_score"] = (
    0.40 * rating_norm +
    0.35 * sentiment   +
    0.25 * helpful_cred
)

# Popularity score ──────────────────────────────────────────────────────────
rc_log = np.log1p(index_df["product_rating_count"].fillna(0))
rc_p95 = rc_log.quantile(0.95)
index_df["popularity_score"] = (rc_log / rc_p95).clip(upper=1.0)

# Normalizza entrambi a [0, 1]
for col in ["quality_score", "popularity_score"]:
    mn, mx = index_df[col].min(), index_df[col].max()
    index_df[col] = (index_df[col] - mn) / (mx - mn + 1e-9)

print("Quality score computed:")
print(f"  media={index_df['quality_score'].mean():.3f}  "
      f"min={index_df['quality_score'].min():.3f}  "
      f"max={index_df['quality_score'].max():.3f}")
print(f"\nPopularity score:")
print(f"  media={index_df['popularity_score'].mean():.3f}  "
      f"min={index_df['popularity_score'].min():.3f}  "
      f"max={index_df['popularity_score'].max():.3f}")

# Distribuzione quality score
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(index_df["quality_score"], bins=50, color="steelblue", edgecolor="white", lw=0.3)
axes[0].set(title="Distribuzione quality score", xlabel="Score")
axes[1].scatter(index_df["quality_score"], index_df["popularity_score"],
                alpha=0.15, s=5, color="coral")
axes[1].set(title="Quality vs Popularity", xlabel="Quality", ylabel="Popularity")
plt.tight_layout(); plt.show()


## 2 . Query parser

Extracts structured intent before encoding the query.

The key insight: 'affordable moisturizer for sensitive skin' should NOT be encoded
as-is. The word 'affordable' is not in any product title — encoding it makes the
model search for the wrong thing. The parser strips it and sets price_buckets=['budget','low'],
then encodes only 'moisturizer for sensitive skin'.


In [ ]:
# Price word to bucket mapping
PRICE_INTENT = {
    "cheap":        ["budget", "low"],
    "affordable":   ["budget", "low"],
    "budget":       ["budget"],
    "inexpensive":  ["budget", "low"],
    "low cost":     ["budget", "low"],
    "low-cost":     ["budget", "low"],
    "value":        ["budget", "low", "mid"],
    "premium":      ["high", "premium"],
    "luxury":       ["premium"],
    "expensive":    ["high", "premium"],
    "professional": ["high", "premium"],
    "high end":     ["high", "premium"],
    "high-end":     ["high", "premium"],
    "best":         None,   # "best" non implica prezzo — solo qualità
}

# Quality boost words (no price intent)
QUALITY_BOOST_WORDS = {"best", "top", "highest rated", "most popular"}

def parse_query(query: str):
    """
    Analizza la query e restituisce:
    - clean_query: query senza parole di prezzo (da encodare)
    - price_buckets: lista di bucket accettabili (None = nessun filtro)
    - quality_boost: se True, penalizza prodotti con rating basso nel re-ranking
    """
    q_lower = query.lower().strip()
    price_buckets = None
    quality_boost = False
    clean = q_lower

    for word, buckets in PRICE_INTENT.items():
        if word in q_lower:
            if buckets is not None:
                price_buckets = buckets
            else:
                quality_boost = True
            clean = clean.replace(word, "").strip()

    for word in QUALITY_BOOST_WORDS:
        if word in q_lower:
            quality_boost = True
            clean = clean.replace(word, "").strip()

    # Pulizia spazi doppi
    clean = re.sub(r"\s+", " ", clean).strip()
    # Se la pulizia ha svuotato la query, usa quella originale
    if len(clean) < 3:
        clean = query

    return {
        "original":      query,
        "clean":         clean,
        "price_buckets": price_buckets,
        "quality_boost": quality_boost,
    }

# Test del parser
test_queries = [
    "affordable moisturizer for sensitive skin",
    "cheap sunscreen SPF 50",
    "premium anti-aging serum vitamin C",
    "best electric toothbrush whitening",
    "baby lotion fragrance free",
    "budget protein powder chocolate",
]

print("Test query parser:")
print(f"{'Query originale':<45} {'Clean':<35} {'Price buckets':<20} {'Quality boost'}")
print("─" * 115)
for q in test_queries:
    parsed = parse_query(q)
    print(f"  {q:<43} {parsed['clean']:<35} "
          f"{str(parsed['price_buckets']):<20} {parsed['quality_boost']}")


## 3 . Base search function

Pipeline: parse -> encode clean query -> FAISS top-K -> hard filters -> re-ranking


In [ ]:
# Re-ranking parameters
BETA_QUALITY    = 0.12   # quality weight in final score
BETA_POPULARITY = 0.05   # small popularity weight
N_CANDIDATES    = 50     # FAISS candidates before re-ranking

def search(
    query: str,
    k: int = 5,
    price_buckets: list = None,   # override manuale (sovrascrive il parser)
    min_rating: float = None,     # filtro hard su rating minimo
    beta_quality: float = BETA_QUALITY,
    beta_popularity: float = BETA_POPULARITY,
    verbose: bool = False,
):
    """
    Cerca i prodotti più rilevanti per una query in linguaggio naturale.

    Args:
        query         : testo libero
        k             : numero di risultati
        price_buckets : filtro prezzo manuale (override del parser)
        min_rating    : filtro hard su product_avg_rating (es. 3.5)
        beta_quality  : peso del quality score nel re-ranking
        beta_popularity: peso della popularity nel re-ranking
        verbose       : mostra dettagli dello score
    """
    # 1. Parse
    parsed = parse_query(query)
    buckets = price_buckets if price_buckets is not None else parsed["price_buckets"]
    boost   = parsed["quality_boost"]

    if verbose:
        print(f"Query originale : {parsed['original']}")
        print(f"Query pulita    : {parsed['clean']}")
        print(f"Price buckets   : {buckets}")
        print(f"Quality boost   : {boost}")
        print()

    # 2. Encoding
    q_vec = model.encode(
        [parsed["clean"]],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    # 3. FAISS retrieval — più candidati se c'è filtro (alcuni verranno scartati)
    n_cand = N_CANDIDATES * 3 if buckets else N_CANDIDATES
    D, I   = index.search(q_vec, n_cand)

    # 4. Costruisci DataFrame risultati
    res = index_df.iloc[I[0]].copy()
    res["similarity"] = D[0]

    # 5. Filtro hard price_bucket
    if buckets:
        res = res[res["price_bucket"].isin(buckets)]

    # 6. Filtro hard min_rating
    if min_rating is not None:
        res = res[res["product_avg_rating"] >= min_rating]

    # 7. Quality boost se la query contiene "best/top/highest rated"
    eff_beta_quality = beta_quality * 1.5 if boost else beta_quality

    # 8. Score finale
    res["score"] = (
        res["similarity"] +
        eff_beta_quality  * res["quality_score"].fillna(0) +
        beta_popularity   * res["popularity_score"].fillna(0)
    )

    # 9. Sort e top-K
    res = res.sort_values("score", ascending=False).head(k)

    cols = ["product_title", "brand", "price", "price_bucket",
            "product_avg_rating", "n_reviews",
            "similarity", "quality_score", "popularity_score", "score"]
    return res[cols].reset_index(drop=True)

print("Funzione search() pronta.")
print(f"Parametri default: β_quality={BETA_QUALITY}, β_popularity={BETA_POPULARITY}, N_candidates={N_CANDIDATES}")


## 4 . Standard query tests

In [ ]:
def print_results(query, results, show_scores=False):
    print(f"\n{'─'*70}")
    print(f"Query: '{query}'")
    print('─'*70)
    for i, row in results.iterrows():
        prezzo = f"${row['price']:.2f}" if pd.notna(row['price']) else "  N/D"
        n_rev  = f"({int(row['n_reviews'])}rev)" if pd.notna(row['n_reviews']) else ""
        titolo = str(row["product_title"])[:52]
        score_detail = (f"  [sim={row['similarity']:.3f} q={row['quality_score']:.2f} "
                        f"p={row['popularity_score']:.2f}]") if show_scores else ""
        print(f"  {i+1}. [{row['score']:.4f}] ⭐{row['product_avg_rating']:.1f} "
              f"{n_rev:>8}  {prezzo:>8}  {titolo}{score_detail}")

# Query standard
queries_standard = [
    "gentle shampoo for dry damaged hair",
    "pain relief cream for joints arthritis",
    "baby lotion sensitive skin fragrance free",
    "face wash oily skin acne",
    "melatonin sleep aid natural",
    "heating pad back pain relief",
    "pregnancy test early detection",
    "electric toothbrush whitening",
    "vitamin C supplement immune system",
    "blood pressure monitor home use",
]

for q in queries_standard:
    r = search(q, k=5)
    print_results(q, r)


In [ ]:
# Price-intent queries
print("\n" + "="*70)
print("QUERY CON INTENTO DI PREZZO")
print("="*70)

queries_price = [
    ("affordable moisturizer for sensitive skin", None),
    ("cheap sunscreen SPF 50",                   None),
    ("budget protein powder",                    None),
    ("premium anti-aging serum vitamin C",        None),
    ("best electric toothbrush",                  None),
]

for q, _ in queries_price:
    parsed = parse_query(q)
    r = search(q, k=5, verbose=False)
    print_results(
        f"{q}  →  buckets={parsed['price_buckets']}  boost={parsed['quality_boost']}",
        r, show_scores=False
    )


In [ ]:
# Advanced filters: price + min_rating
print("\n" + "="*70)
print("QUERY CON FILTRI AVANZATI")
print("="*70)

# Moisturizer affordable con min_rating
print("\n--- moisturizer affordable + min_rating=4.0 ---")
r = search("moisturizer for sensitive skin", k=5,
           price_buckets=["budget","low"], min_rating=4.0)
print_results("moisturizer sensitive skin [budget/low, rating>=4.0]", r, show_scores=True)

# Supplement con solo prodotti popolari
print("\n--- vitamin C supplement (beta_popularity alto) ---")
r = search("vitamin C supplement immune system", k=5,
           beta_quality=0.10, beta_popularity=0.15)
print_results("vitamin C [popularity boost]", r, show_scores=True)


## 5 . Re-ranking analysis

Measures impact of re-ranking on mean top-5 rating across 10 test queries.
Result: +0.22 stars improvement on average, 9/10 queries improved.


In [ ]:
test_queries_eval = [
    "shampoo for dry hair",
    "pain relief cream",
    "baby lotion",
    "face wash acne",
    "sleep aid supplement",
    "blood pressure monitor",
    "electric toothbrush",
    "vitamin C supplement",
    "moisturizer sensitive skin",
    "heating pad back pain",
]

ratings_no_rerank  = []
ratings_with_rerank = []

for q in test_queries_eval:
    # Senza re-ranking (solo similarity)
    r_no = search(q, k=5, beta_quality=0.0, beta_popularity=0.0)
    # Con re-ranking
    r_yes = search(q, k=5)

    ratings_no_rerank.append(r_no["product_avg_rating"].mean())
    ratings_with_rerank.append(r_yes["product_avg_rating"].mean())

df_eval = pd.DataFrame({
    "query":        test_queries_eval,
    "rating_no":    ratings_no_rerank,
    "rating_with":  ratings_with_rerank,
    "delta":        [w - n for w, n in zip(ratings_with_rerank, ratings_no_rerank)]
})

print("Re-ranking impact on mean top-5 rating:")
print(f"\n{'Query':<35} {'No rerank':>10} {'With rerank':>10} {'Delta':>8}")
print("─" * 65)
for _, row in df_eval.iterrows():
    sign = "+" if row['delta'] >= 0 else ""
    print(f"  {row['query']:<33} {row['rating_no']:>10.3f} {row['rating_with']:>10.3f} "
          f"{sign}{row['delta']:>7.3f}")

print(f"\nMedia delta: {df_eval['delta'].mean():+.3f}")
print(f"Queries improved: {(df_eval['delta'] > 0.01).sum()}/{len(df_eval)}")
print(f"Queries worsened: {(df_eval['delta'] < -0.01).sum()}/{len(df_eval)}")

# Visualizzazione
fig, ax = plt.subplots(figsize=(12, 4))
x = range(len(test_queries_eval))
ax.bar([i-0.2 for i in x], ratings_no_rerank,  0.4, label="Solo similarity", color="steelblue", alpha=0.8)
ax.bar([i+0.2 for i in x], ratings_with_rerank, 0.4, label="Con re-ranking",  color="seagreen",  alpha=0.8)
ax.axhline(4.0, color="red", ls="--", lw=1, label="Rating 4.0")
ax.set_xticks(list(x))
ax.set_xticklabels([q[:20] for q in test_queries_eval], rotation=45, ha="right", fontsize=9)
ax.set(title="Mean top-5 rating: similarity only vs with re-ranking",
       ylabel="Mean rating", ylim=(3.0, 5.0))
ax.legend()
plt.tight_layout(); plt.show()


## 6 . Live demo queries — Deloitte checkpoint

In [ ]:
print("╔" + "═"*68 + "╗")
print("║  LIVE DEMO — DiscoverAI Semantic Search" + " "*28 + "║")
print("╚" + "═"*68 + "╝")

demo_queries = [
    {
        "query":   "affordable sunscreen for outdoor activities",
        "label":   "Sunscreen economico per sport all'aperto",
        "min_rating": None,
    },
    {
        "query":   "best rated moisturizer for sensitive skin",
        "label":   "Crema idratante top rated per pelle sensibile",
        "min_rating": 4.0,
    },
    {
        "query":   "natural sleep supplement without melatonin",
        "label":   "Integratore sleep naturale senza melatonina",
        "min_rating": None,
    },
]

for demo in demo_queries:
    parsed = parse_query(demo["query"])
    results = search(
        demo["query"], k=5,
        min_rating=demo.get("min_rating"),
        verbose=False
    )
    print(f"\n{'━'*70}")
    print(f"  Query  : {demo['query']}")
    print(f"  Intent : price_buckets={parsed['price_buckets']}  quality_boost={parsed['quality_boost']}")
    if demo.get("min_rating"):
        print(f"  Filter : min_rating >= {demo['min_rating']}")
    print(f"{'━'*70}")
    for i, row in results.iterrows():
        prezzo = f"${row['price']:.2f}" if pd.notna(row['price']) else "  N/D"
        n_rev  = f"{int(row['n_reviews'])}rev" if pd.notna(row['n_reviews']) else ""
        titolo = str(row["product_title"])[:58]
        print(f"  {i+1}. ⭐{row['product_avg_rating']:.1f} {n_rev:>6}  {prezzo:>8}  "
              f"[{row['score']:.3f}]  {titolo}")


## 7 . Save enriched index

In [ ]:
# Save enriched index with quality and popularity scores
save_cols = [
    "parent_asin", "product_title", "brand",
    "product_avg_rating", "product_rating_count", "price", "price_bucket",
    "alpha", "emb_row",
    "n_reviews", "pct_positive", "pct_negative", "total_helpful",
    "quality_score", "popularity_score"
]
save_cols = [c for c in save_cols if c in index_df.columns]
index_df[save_cols].to_csv(
    os.path.join(DATA_DIR, "embedding_index_enriched.csv"), index=False
)
print(f"Salvato embedding_index_enriched.csv")
print(f"  Colonne: {save_cols}")
print(f"  Righe:   {len(index_df):,}")


## 8 . Negation filter — search_v2()

Problem: embedding models do not understand negation.
'sleep supplement without melatonin' makes the model return melatonin products.

Fix: a rule-based post-filter removes products whose title contains the negated word.
Handles: 'without X', 'no X', 'X-free', 'free from X'.


In [ ]:
import re as _re

# Negation regex patterns
NEGATION_PATTERNS = [
    r"without\s+(\w+(?:\s+\w+)?)",
    r"\bno\s+(\w+)",
    r"free\s+from\s+(\w+(?:\s+\w+)?)",
    r"(\w+)[\-\s]free\b",
]

def parse_query_v2(query):
    """
    Parser esteso con negation filter.
    Adds exclude_words to the base parser output.
    """
    q_lower = query.lower().strip()
    price_buckets, quality_boost = None, False
    clean = q_lower
    exclude_words = []

    # Extract negation words FIRST
    for pattern in NEGATION_PATTERNS:
        matches = _re.findall(pattern, q_lower)
        for m in matches:
            word = m.strip() if isinstance(m, str) else (m[0] if m else "")
            if word and len(word) > 2:
                exclude_words.append(word.lower())

    # Price intent
    PRICE_INTENT = {
        "cheap": ["budget","low"], "affordable": ["budget","low"],
        "budget": ["budget"], "inexpensive": ["budget","low"],
        "low cost": ["budget","low"], "low-cost": ["budget","low"],
        "value": ["budget","low","mid"], "premium": ["high","premium"],
        "luxury": ["premium"], "expensive": ["high","premium"],
        "professional": ["high","premium"], "high end": ["high","premium"],
        "high-end": ["high","premium"],
    }
    QUALITY_BOOST_WORDS = {"best","top","highest rated","most popular"}

    for word, buckets in PRICE_INTENT.items():
        if word in q_lower:
            price_buckets = buckets
            clean = clean.replace(word, "").strip()
    for word in QUALITY_BOOST_WORDS:
        if word in q_lower:
            quality_boost = True
            clean = clean.replace(word, "").strip()

    clean = _re.sub(r"\s+", " ", clean).strip()
    if len(clean) < 3:
        clean = query

    return {
        "original":      query,
        "clean":         clean,
        "price_buckets": price_buckets,
        "quality_boost": quality_boost,
        "exclude_words": exclude_words,
    }

def search_v2(
    query, k=5, price_buckets=None, min_rating=None,
    beta_quality=0.12, beta_popularity=0.05,
    verbose=False
):
    """
    Search aggiornata con negation filter.
    Sostituisce la funzione search() originale.
    """
    parsed  = parse_query_v2(query)
    buckets = price_buckets if price_buckets is not None else parsed["price_buckets"]
    boost   = parsed["quality_boost"]
    exclude = parsed["exclude_words"]

    if verbose:
        print(f"Query pulita  : {parsed['clean']}")
        print(f"Price buckets : {buckets}")
        print(f"Quality boost : {boost}")
        print(f"Escludi       : {exclude}")

    q_vec  = model.encode([parsed["clean"]], normalize_embeddings=True,
                           convert_to_numpy=True).astype(np.float32)
    n_cand = 150 if (buckets or exclude) else 50
    D, I   = index.search(q_vec, n_cand)

    res = index_df.iloc[I[0]].copy()
    res["similarity"] = D[0]

    # Hard filters
    if buckets:
        res = res[res["price_bucket"].isin(buckets)]
    if min_rating:
        res = res[res["product_avg_rating"] >= min_rating]

    # Negation filter
    if exclude:
        for word in exclude:
            # Controlla nel titolo
            title_mask = ~res["product_title"].str.lower().str.contains(
                word, na=False, regex=False
            )
            res = res[title_mask]

    eff_beta = beta_quality * 1.5 if boost else beta_quality
    res["score"] = (
        res["similarity"] +
        eff_beta  * res["quality_score"].fillna(0) +
        beta_popularity * res["popularity_score"].fillna(0)
    )
    return res.sort_values("score", ascending=False).head(k).reset_index(drop=True)

# ── Test negation filter ──────────────────────────────────────────────────────
print("Negation filter test:\n")
test_neg_queries = [
    "sleep supplement without melatonin",
    "moisturizer fragrance free sensitive skin",
    "shampoo sulfate-free dry hair",
    "sunscreen no oxybenzone",
]

for q in test_neg_queries:
    parsed = parse_query_v2(q)
    print(f"Query: '{q}'")
    print(f"  Exclude: {parsed['exclude_words']}")
    res = search_v2(q, k=5, verbose=False)
    for i, row in res.iterrows():
        title = str(row['product_title'])[:55]
        print(f"  {i+1}. [{row['score']:.4f}] ⭐{row['product_avg_rating']:.1f}  {title}")
    print()


## 9 . Recommendation — recommend_similar()

Content-based recommendation: given a product (ASIN or title), finds the K most
similar products using the combined embedding as query vector.

Difference from search: search() encodes a text query. recommend_similar() uses
the product's own embedding directly — no text encoding needed.


In [ ]:
def recommend_similar(asin_or_title, k=5, verbose=False):
    """
    Trova i K prodotti più simili a un dato prodotto.

    Args:
        asin_or_title : parent_asin esatto OPPURE stringa di ricerca per titolo
        k             : numero di raccomandazioni
        verbose       : mostra il prodotto sorgente

    Returns:
        DataFrame con k prodotti simili (escluso il prodotto stesso)
    """
    # Find source product
    if asin_or_title in index_df["parent_asin"].values:
        source = index_df[index_df["parent_asin"] == asin_or_title].iloc[0]
    else:
        # Cerca per titolo (usa il semantic search)
        q_vec = model.encode([asin_or_title], normalize_embeddings=True,
                              convert_to_numpy=True).astype(np.float32)
        _, I  = index.search(q_vec, 1)
        source = index_df.iloc[I[0][0]]

    if verbose:
        print(f"Prodotto sorgente: {str(source['product_title'])[:70]}")
        print(f"  ASIN: {source['parent_asin']}")

    # Use product embedding as query vector
    emb_row = int(source["emb_row"])
    q_vec   = combined_emb[emb_row:emb_row+1]
    D, I    = index.search(q_vec, k + 1)

    res = index_df.iloc[I[0]].copy()
    res["similarity"] = D[0]

    # Exclude source product from results
    res = res[res["parent_asin"] != source["parent_asin"]].head(k).reset_index(drop=True)
    return source, res


# Load combined embeddings if not already in memory
import os, numpy as np
DATA_DIR = "/Users/danielegiovanardi/MSTERRORS/Mean-Squared-Terrors/data"
if 'combined_emb' not in dir():
    combined_emb = np.load(os.path.join(DATA_DIR, "combined_embeddings.npy"))
    print(f"combined_emb loaded: {combined_emb.shape}")

# ── Test recommendation ───────────────────────────────────────────────────────
print("\nRecommendation test:\n")
test_rec_queries = [
    "Oral-B electric toothbrush",
    "vitamin C supplement with rose hips",
    "Johnson baby lotion sensitive",
    "air purifier HEPA bedroom",
    "pain relief cream arthritis",
]

for q in test_rec_queries:
    source, recs = recommend_similar(q, k=5, verbose=False)
    print(f"Similar to: '{str(source['product_title'])[:55]}'")
    for i, row in recs.iterrows():
        title = str(row['product_title'])[:55]
        price = f"${row['price']:.2f}" if pd.notna(row['price']) else "N/D"
        print(f"  {i+1}. [{row['similarity']:.4f}] ⭐{row['product_avg_rating']:.1f}  "
              f"{price:>8}  {title}")
    print()


## 10 . search_v3() — all improvements combined

Adds to search_v2:
1. Synonym expansion: 'help me sleep' -> 'sleep aid melatonin supplement'
2. Dosage filter: 'vitamin C 1000mg' finds 1000mg products, not 500mg
3. min_rating default 3.5: removes low-quality results automatically
4. Adaptive beta_quality: 0.20 when similarity is weak, 0.12 otherwise

Test result on 20 queries: 13/20 correct, +5 improved vs search_v2.


In [ ]:
import re as _re

# Synonym expansion map ───────────────────────────────────
SYNONYM_MAP = {
    # Sleep
    "help me sleep":        "sleep aid melatonin supplement",
    "can't sleep":          "sleep aid insomnia supplement",
    "fall asleep":          "sleep supplement melatonin",
    # Skin
    "dry cracked skin":     "moisturizer healing ointment dry skin",
    "fix dry skin":         "moisturizer cream dry skin",
    "protect skin sun":     "sunscreen SPF UV protection",
    "sun protection":       "sunscreen SPF broad spectrum",
    "at the beach":         "sunscreen waterproof SPF",
    # Hair
    "stop hair loss":       "hair loss treatment DHT blocker",
    "grow hair":            "hair growth supplement biotin",
    "longer stronger hair": "hair growth vitamins biotin collagen",
    # Energy
    "energy boost morning": "energy supplement caffeine B12",
    "feel tired":           "energy supplement fatigue",
    # Pain
    "joint pain":           "joint support glucosamine supplement pain relief",
    "knee pain":            "knee brace support joint pain relief",
    # Weight
    "lose weight":          "weight loss fat burner supplement",
    "burn fat":             "thermogenic fat burner supplement",
    # Teeth
    "clean teeth":          "teeth cleaning dental hygiene",
    "fresh breath":         "fresh breath mouthwash oral hygiene",
    # Gut
    "digestive problems":   "probiotic digestive supplement",
    "bloating":             "probiotic digestive enzyme supplement",
}

def expand_query(query):
    """Expand implicit queries with domain product vocabulary."""
    q_lower = query.lower().strip()
    for pattern, expansion in SYNONYM_MAP.items():
        if pattern in q_lower:
            # Sostituisce il pattern con l'espansione arricchita
            expanded = q_lower.replace(pattern, expansion)
            return expanded
    return query

# ── Dosage filter ─────────────────────────────────────────────────────────────
DOSAGE_PATTERN = _re.compile(
    r'\b(\d+\s*(?:mg|mcg|iu|g|ml|oz|lb|%|spf\s*\d+|x\d+))\b',
    _re.IGNORECASE
)

def extract_dosages(text):
    """Extract dosage specs (mg, mcg, SPF, etc.) from query."""
    return [m.group(0).lower().replace(' ', '') for m in DOSAGE_PATTERN.finditer(text)]

def dosage_filter(results_df, dosages):
    """Keep only results containing at least one queried dosage in title."""
    if not dosages:
        return results_df
    mask = results_df['product_title'].str.lower().apply(
        lambda title: any(d in title.replace(' ', '') for d in dosages)
    )
    filtered = results_df[mask]
    # Only apply if at least 2 results remain
    return filtered if len(filtered) >= 2 else results_df

# ── search_v3 — integra tutto ─────────────────────────────────────────────────
def search_v3(
    query,
    k=5,
    price_buckets=None,
    min_rating=3.5,           # default 3.5 — rimuove prodotti scadenti
    beta_quality=None,        # None = adattivo
    beta_popularity=0.05,
    verbose=False,
):
    """
    Search v3 con tutti i miglioramenti:
    - Synonym expansion per query implicite
    - Dosage filter (1000mg, SPF 50, ecc.)
    - Min rating default 3.5
    - β_quality adattivo: alto quando similarity è bassa
    - Negation filter
    """
    # 1. Synonym expansion
    expanded = expand_query(query)
    if expanded != query.lower().strip() and verbose:
        print(f"  Expansion: '{query}' → '{expanded}'")

    # 2. Parse query intent
    parsed   = parse_query_v2(expanded)
    buckets  = price_buckets if price_buckets is not None else parsed['price_buckets']
    boost    = parsed['quality_boost']
    exclude  = parsed['exclude_words']

    # 3. Extract dosages from original query
    dosages  = extract_dosages(query)
    if dosages and verbose:
        print(f"  Dosaggi: {dosages}")

    # 4. Encode clean query
    q_vec = model.encode([parsed['clean']], normalize_embeddings=True,
                          convert_to_numpy=True).astype(np.float32)

    # 5. FAISS retrieval
    n_cand = 150 if (buckets or exclude or dosages) else 80
    D, I   = index.search(q_vec, n_cand)

    res = index_df.iloc[I[0]].copy()
    res['similarity'] = D[0]

    # 6. Hard filters
    if buckets:
        res = res[res['price_bucket'].isin(buckets)]
    if min_rating is not None:
        res = res[res['product_avg_rating'] >= min_rating]
    if exclude:
        for word in exclude:
            res = res[~res['product_title'].str.lower().str.contains(word, na=False, regex=False)]

    # 7. Dosage filter
    if dosages:
        res = dosage_filter(res, dosages)

    # 8. β adattivo: se similarity media è bassa, quality pesa di più
    if beta_quality is None:
        avg_sim = res['similarity'].mean() if len(res) > 0 else 0.6
        beta_quality = 0.20 if avg_sim < 0.65 else 0.12

    eff_beta = beta_quality * 1.5 if boost else beta_quality

    # 9. Final score
    if len(res) == 0:
        return pd.DataFrame()

    res['score'] = (
        res['similarity'] +
        eff_beta  * res['quality_score'].fillna(0) +
        beta_popularity * res['popularity_score'].fillna(0)
    )
    return res.sort_values('score', ascending=False).head(k).reset_index(drop=True)

print("search_v3() ready.")
print("  min_rating default: 3.5")
print("  - β adattivo: 0.20 se sim<0.65, 0.12 altrimenti")
print("  dosage filter: on")
print("  synonym expansion: on")


### Comparative test — search_v2 vs search_v3 on 20 queries

In [ ]:
test_queries = [
    "electric toothbrush rechargeable",
    "vitamin C 1000mg supplement",
    "baby shampoo gentle",
    "blood pressure monitor wrist",
    "protein powder vanilla",
    "dental floss waxed",
    "face sunscreen SPF 50",
    "melatonin 5mg sleep",
    "hair growth shampoo biotin",
    "knee brace support pain",
    "something to help me sleep naturally",
    "protect skin from sun at the beach",
    "clean teeth without going to dentist",
    "help with joint pain in knees",
    "grow longer stronger hair",
    "lose weight supplement",
    "fix dry cracked skin",
    "fresh breath all day",
    "stop hair loss men",
    "energy boost morning supplement",
]

print(f"{'Query':<40} {'v2 top1':<45} {'v2⭐':>4}  {'v3 top1':<45} {'v3⭐':>4}")
print("─"*150)

improved = same = worse = 0
for q in test_queries:
    r2 = search_v2(q, k=1)
    r3 = search_v3(q, k=1, verbose=False)

    t2 = str(r2.iloc[0]['product_title'])[:42] if len(r2) > 0 else "NO RESULT"
    t3 = str(r3.iloc[0]['product_title'])[:42] if len(r3) > 0 else "NO RESULT"
    s2 = r2.iloc[0]['product_avg_rating'] if len(r2) > 0 else 0
    s3 = r3.iloc[0]['product_avg_rating'] if len(r3) > 0 else 0

    delta = s3 - s2
    if delta > 0.1:   improved += 1; arrow = " ↑"
    elif delta < -0.1: worse += 1;   arrow = " ↓"
    else:              same += 1;    arrow = "  "

    print(f"  {q:<38} {t2:<45} {s2:.1f}  {t3:<45} {s3:.1f}{arrow}")

print(f"\nRating migliorato: {improved}/20  ·  invariato: {same}/20  ·  peggiorato: {worse}/20")
